# Image Classification vs. Object Detection vs. Image Segmentation

## 1. Image Classification

**Definition:**  
Image Classification is the task of assigning a label (or class) to an entire image.

**Output:**  
A single label for the whole image.

**Use Case Example:**  
- Given an image, classify whether it is a **cat** or a **dog**.


---

## 2. Object Detection

**Definition:**  
Object Detection locates and classifies multiple objects in an image by drawing bounding boxes around them.

**Output:**  
Bounding boxes with class labels and confidence scores.

**Use Case Example:**  
- Detect all **cars** and **pedestrians** in a traffic image.


---

## 3. Image Segmentation

**Definition:**  
Image Segmentation classifies each pixel in the image to determine object boundaries.

**Types:**
- **Semantic Segmentation:** Classifies each pixel (e.g., all cats = same class).
- **Instance Segmentation:** Detects separate objects of the same class (e.g., two different cats).

**Output:**  
A pixel-wise mask showing which parts belong to which object/class.

**Use Case Example:**  
- Segment roads, cars, and pedestrians for self-driving cars.

---

## Summary Table

| Feature               | Image Classification | Object Detection      | Image Segmentation         |
|-----------------------|----------------------|------------------------|-----------------------------|
| Goal                  | Label whole image    | Detect & label objects | Label each pixel            |
| Output                | Single class         | Bounding boxes + labels| Pixel-wise mask             |
| Complexity            | Low                  | Medium                 | High                        |
| Example Task          | Is this a cat?       | Where are the cars?    | Which pixels are cars?      |
| Output Visualization  | Single label         | Boxes on image         | Colored pixel regions       |


# Relationship Between Input Image Size and Output Size in Image Segmentation Models

## Key Concept:

In **Image Segmentation**, the goal is to produce an **output mask** that aligns with the **input image**, where each pixel is classified into a category.

---

## 1. **Ideal Case: Same Size Input & Output**

Most modern segmentation models (e.g., **UNet**, **DeepLabV3+**, **FCN**) are designed to output a segmentation mask of the **same spatial dimensions** as the input image.

- **Input:** 256×256 RGB Image  
- **Output:** 256×256×C Mask (where `C` = number of classes)

This alignment ensures pixel-level correspondence between the input and predicted class.

---

## 2. **Why Size Might Differ**

During training and inference:
- **Downsampling** (via pooling or strided convolutions) reduces spatial resolution.
- **Upsampling** (via transposed convolution or interpolation) restores the size.

Without proper upsampling, the output may be smaller.

| Layer Type         | Effect on Size          |
|--------------------|--------------------------|
| Conv2D             | Can maintain or shrink   |
| MaxPooling         | Shrinks size             |
| Transposed Conv    | Enlarges size            |
| Interpolation      | Enlarges size            |
| Skip Connections   | Helps recover details    |

---

## 3. **Maintaining Input-Output Size**

To keep input and output sizes the same:

- Use **padding='same'** in convolutions.
- Add upsampling layers to restore dimensions.
- Use architectures like **UNet**, which combine encoder-decoder paths with skip connections.

---

## 4. **Summary Table**

| Model Architecture | Input Size      | Output Size     | Pixel-wise Matching? |
|--------------------|------------------|------------------|------------------------|
| UNet               | 512×512          | 512×512          | ✅ Yes                 |
| FCN                | 224×224          | 224×224          | ✅ Yes                 |
| DeepLabV3          | 513×513          | 513×513          | ✅ Yes (via upsampling)|
| CNN Classifier     | 224×224          | 1×1 (class only) | ❌ Not applicable      |

---

## 5. **Conclusion**

- **Same Size:** Most segmentation models are designed to output masks matching the input size.
- **If Not:** Upsampling is used to restore original size.
- **Purpose:** Ensures **pixel-wise classification** for accurate segmentation.


# What Are Segmentation Masks in Segmentation Tasks?

## 📌 Definition:

A **segmentation mask** is an image-like matrix where **each pixel** represents the **class label** of the corresponding pixel in the original image.

---

## 🧠 Purpose:

- To perform **pixel-wise classification**.
- Each pixel in the image is labeled as belonging to a particular class (e.g., background, car, person).

---

## 🖼️ Example:

### Original Image:
An image with a dog and a cat.

### Segmentation Mask:
- Background → 0 (black)  
- Dog → 1 (gray)  
- Cat → 2 (white)

```text
[
 [0, 0, 0, 1, 1],
 [0, 2, 2, 1, 1],
 [0, 2, 2, 0, 0]
]
```


---

## 🎯 Types of Segmentation Masks:

| Type                   | Description                                                    |
|------------------------|----------------------------------------------------------------|
| **Binary Mask**        | Pixels are either 0 (background) or 1 (object of interest)     |
| **Multiclass Mask**    | Each pixel is assigned a class label (0, 1, 2, ..., N)         |
| **Instance Mask**      | Separates different instances of the same class (e.g., two dogs)|

---

## 🔍 Use in Training:

- Ground truth masks are used as labels.
- Model learns to predict the correct class for each pixel.
- Loss functions like **Cross-Entropy** or **Dice Loss** compare predicted masks to true masks.

---

## ✅ Summary:

- A **segmentation mask** is a pixel-level label image.
- It tells the model what each pixel represents.
- Essential for **semantic** and **instance segmentation** tasks.


# 🎯 Evaluation Metrics & Loss Functions in Image Segmentation

---

## 🔄 Dice Score vs IoU Score

| Metric       | Formula                                                                 | Range     | Meaning                                                                 |
|--------------|-------------------------------------------------------------------------|-----------|-------------------------------------------------------------------------|
| **Dice Score (F1 Score)** | \( \text{Dice} = \frac{2 \times |A \cap B|}{|A| + |B|} \)             | 0 to 1   | Measures overlap between predicted and true mask (more sensitive to small regions) |
| **IoU (Jaccard Index)**   | \( \text{IoU} = \frac{|A \cap B|}{|A \cup B|} \)                      | 0 to 1   | Measures ratio of intersection over union between predicted and true mask |

> **A** = Predicted Mask  
> **B** = Ground Truth Mask

### ✅ Key Differences:
- **Dice** emphasizes overlap twice as much as size.
- **IoU** penalizes over-prediction and under-prediction more evenly.
- Dice is often slightly higher than IoU for the same prediction.

---

## 🧪 Common Evaluation Metrics in Segmentation

| Metric            | Use Case                                  |
|-------------------|--------------------------------------------|
| **IoU Score**     | General-purpose segmentation accuracy      |
| **Dice Coefficient** | Medical image segmentation, imbalanced data |
| **Pixel Accuracy** | Percentage of correctly classified pixels |
| **Precision/Recall** | For multi-class segmentation problems     |

---

## 💡 Common Loss Functions in Segmentation

| Loss Function           | When to Use                                             |
|--------------------------|----------------------------------------------------------|
| **Binary Cross-Entropy (BCE)** | For binary segmentation (foreground vs background)     |
| **Categorical Cross-Entropy** | For multi-class segmentation                         |
| **Dice Loss**            | For imbalanced data, especially small objects            |
| **IoU Loss**             | Encourages better shape prediction                        |
| **Tversky Loss**         | Customizable Dice-like loss with precision/recall control |
| **Combo Loss**           | Combines Dice + BCE or IoU + CrossEntropy                 |
| **Focal Loss**           | For handling hard-to-classify pixels and class imbalance  |

---

## 🔁 Summary

- **IoU** and **Dice** both measure overlap but in slightly different ways.
- Use **Dice Loss** or **IoU Loss** when class imbalance exists.
- Always choose metrics/loss functions based on data characteristics and goals.
